In [1]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver

import sqlite3

@tool
def write_file(filename: str, content: str) -> str:
    """
    Write content into a file.
    """
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)

    return f"File '{filename}' written successfully."


@tool
def execute_sql(query: str) -> str:
    """
    Execute an SQL query on the local SQLite database.
    """
    conn = sqlite3.connect("data.sqlite")
    cursor = conn.cursor()

    try:
        cursor.execute(query)

        # If SELECT query
        if query.strip().lower().startswith("select"):
            rows = cursor.fetchall()
            conn.close()
            return str(rows)

        # Otherwise commit changes
        conn.commit()
        conn.close()

        return "SQL query executed successfully."

    except Exception as e:
        conn.close()
        return f"SQL Error: {str(e)}"

@tool
def read_data(filename: str) -> str:
    """
    Read text data from a file.
    """
    try:
        with open(filename, "r", encoding="utf-8") as f:
            return f.read()

    except Exception as e:
        return f"Read Error: {str(e)}"


In [2]:
agent = create_agent(
    model="gpt-5.4",

    tools=[
        write_file,
        execute_sql,
        read_data
    ],

    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": True,
                "execute_sql": {
                    "allowed_decisions": [
                        "approve",
                        "reject"
                    ]
                },

                "read_data": False,
            },

            description_prefix="Tool execution pending approval",
        ),
    ],

    checkpointer=MemorySaver(),
)

In [3]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Create a file named test.txt containing 'Hello World'"
        }
    ]
},config={
    "configurable":{"thread_id":3}
}
)

In [12]:
response["__interrupt__"][0]

Interrupt(value={'action_requests': [{'name': 'write_file', 'args': {'filename': 'test.txt', 'content': 'Hello World'}, 'description': "Tool execution pending approval\n\nTool: write_file\nArgs: {'filename': 'test.txt', 'content': 'Hello World'}"}], 'review_configs': [{'action_name': 'write_file', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='f8d57aac3b46c84d5673ae0583cd746a')

In [11]:
from langgraph.types import Command

# Resume the agent with an approval decision
agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]} 
    ),
    config={
        "configurable":{"thread_id":3}
    }, # Same thread ID to resume the paused conversation
)

{'messages': [HumanMessage(content="Create a file named test.txt containing 'Hello World'", additional_kwargs={}, response_metadata={}, id='a2b0cac2-59f1-4829-b7a4-884c44ea6b1b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 184, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-ENb0qVivjVgcEIqoTeRuHddpWfgdt', 'service_tier': 'priority', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a09a25-40aa-7120-9094-623e4fca9465-0', tool_calls=[{'name': 'write_file', 'args': {'filename': 'test.txt', 'content': 'Hello W

In [17]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Create a file named test.txt containing 'Hello World'"
        }
    ]
},config={
    "configurable":{"thread_id":3}
}
)



In [18]:
response

{'messages': [HumanMessage(content="Create a file named test.txt containing 'Hello World'", additional_kwargs={}, response_metadata={}, id='9f2609a6-e148-4c8d-8b4a-4cb36bf4c672'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 184, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DenlK1Mul6YlbfQpuuWxDUb9f3f6w', 'service_tier': 'priority', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1dcb-33dd-71e0-865c-bcf7feb01777-0', tool_calls=[{'name': 'write_file', 'args': {'filename': 'test.txt', 'content': 'Hello World'}, 'id': 'call_F2yOBQI0T7CgM9NcumfJdp9U', 'type': 'tool_call'}], invalid_tool_calls=[],

In [19]:
agent.invoke(
    Command(
        resume={"decisions": [{"type": "reject"}]} 
    ),
    config={
        "configurable":{"thread_id":3}
    }, # Same thread ID to resume the paused conversation
)

{'messages': [HumanMessage(content="Create a file named test.txt containing 'Hello World'", additional_kwargs={}, response_metadata={}, id='9f2609a6-e148-4c8d-8b4a-4cb36bf4c672'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 184, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DenlK1Mul6YlbfQpuuWxDUb9f3f6w', 'service_tier': 'priority', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1dcb-33dd-71e0-865c-bcf7feb01777-0', tool_calls=[{'name': 'write_file', 'args': {'filename': 'test.txt', 'content': 'Hello World'}, 'id': 'call_F2yOBQI0T7CgM9NcumfJdp9U', 'type': 'tool_call'}], invalid_tool_calls=[],

In [20]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Create users table"
        }
    ]
},config={
    "configurable":{"thread_id":3}
}
)

In [21]:
response

{'messages': [HumanMessage(content="Create a file named test.txt containing 'Hello World'", additional_kwargs={}, response_metadata={}, id='9f2609a6-e148-4c8d-8b4a-4cb36bf4c672'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 184, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DenlK1Mul6YlbfQpuuWxDUb9f3f6w', 'service_tier': 'priority', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1dcb-33dd-71e0-865c-bcf7feb01777-0', tool_calls=[{'name': 'write_file', 'args': {'filename': 'test.txt', 'content': 'Hello World'}, 'id': 'call_F2yOBQI0T7CgM9NcumfJdp9U', 'type': 'tool_call'}], invalid_tool_calls=[],

In [22]:
agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]} 
    ),
    config={
        "configurable":{"thread_id":3}
    }, # Same thread ID to resume the paused conversation
)

{'messages': [HumanMessage(content="Create a file named test.txt containing 'Hello World'", additional_kwargs={}, response_metadata={}, id='9f2609a6-e148-4c8d-8b4a-4cb36bf4c672'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 184, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DenlK1Mul6YlbfQpuuWxDUb9f3f6w', 'service_tier': 'priority', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1dcb-33dd-71e0-865c-bcf7feb01777-0', tool_calls=[{'name': 'write_file', 'args': {'filename': 'test.txt', 'content': 'Hello World'}, 'id': 'call_F2yOBQI0T7CgM9NcumfJdp9U', 'type': 'tool_call'}], invalid_tool_calls=[],